# Pandas 入门：从财务表到行业分析

我们有两张表：

- `financial_data_example.xlsx`：公司年度财务数据。
- `company_info_example.xlsx`：公司基本信息。

本节不按函数顺序讲 pandas，而是围绕三个阶段目标展开。

## 阶段 1：整理财务表，找出 2020 年表现较好的公司

第一阶段只使用财务表。目标不是“学会读取和筛选”，而是把一张略乱的财务表整理成可以排序和比较的公司名单。

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 20)

def read_code(x):
    return str(x).strip().zfill(6)

finance_raw = pd.read_excel(
    "data/financial_data_example.xlsx",
    converters={"证券代码": read_code},
)

finance_raw.head()

,证券代码,证券简称,统计截止日期,年份,总资产,总负债,净资产,营业收入,净利润,资产负债率
0,000001,平安银行,2018-12-31,2018,3418592000000,3.178550e+12,2.400420e+11,1.062120e+11,24818000000,0.9298
1,000001,平安银行,2019-12-31,2019,3939070000000,3.626087e+12,3.129830e+11,1.268140e+11,28195000000,0.9205
2,000001,平安银行,2020-12-31,2020,4468514000000,4.104383e+12,3.641310e+11,1.432420e+11,28928000000,0.9185
3,000002,万科A,2018-12-31,2018,1528579356474.810059,1.292959e+12,2.356207e+11,2.976793e+11,49272294534.610001,0.8459
4,000002,万科A,2019-12-31,2019,1729929450401.22998,1.459350e+12,2.705791e+11,3.678939e+11,55131614572.089996,0.8436


先判断这张表的规模、字段、数据类型和明显问题。

In [2]:
print("行列数：", finance_raw.shape)
print("列名：", finance_raw.columns.tolist())

finance_raw.info()

行列数： (89, 10)
列名： ['证券代码', '证券简称', '统计截止日期', '年份', '总资产', '总负债', '净资产', '营业收入', '净利润', '资产负债率']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89 entries, 0 to 88
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   证券代码    89 non-null     object        
 1   证券简称    89 non-null     object        
 2   统计截止日期  89 non-null     datetime64[ns]
 3   年份      89 non-null     int64         
 4   总资产     89 non-null     object        
 5   总负债     89 non-null     float64       
 6   净资产     89 non-null     float64       
 7   营业收入    88 non-null     float64       
 8   净利润     89 non-null     object        
 9   资产负债率   89 non-null     float64       
dtypes: datetime64[ns](1), float64(4), int64(1), object(4)
memory usage: 7.1+ KB


In [3]:
finance_raw.isna().sum()

证券代码      0
证券简称      0
统计截止日期    0
年份        0
总资产       0
总负债       0
净资产       0
营业收入      1
净利润       0
资产负债率     0
dtype: int64

财务表里有重复行，也有数字列混入文本。先做一份工作副本，逐步清洗。

In [4]:
finance = finance_raw.copy()

print("重复行数量：", finance.duplicated().sum())
finance = finance.drop_duplicates()

num_cols = ["总资产", "总负债", "净资产", "营业收入", "净利润", "资产负债率"]

for col in num_cols:
    finance[col] = (
        finance[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .replace({"--": np.nan, "nan": np.nan})
    )
    finance[col] = pd.to_numeric(finance[col], errors="coerce")

finance[num_cols].isna().sum()

重复行数量： 1


总资产      0
总负债      0
净资产      0
营业收入     1
净利润      1
资产负债率    0
dtype: int64

把日期转成真正的日期，并基于原有列构造几个更容易解释的指标。

In [5]:
finance["统计截止日期"] = pd.to_datetime(finance["统计截止日期"], errors="coerce")
finance["年份"] = finance["统计截止日期"].dt.year

finance["总资产_亿元"] = finance["总资产"] / 1e8
finance["营业收入_亿元"] = finance["营业收入"] / 1e8
finance["净利率"] = finance["净利润"] / finance["营业收入"]
finance["资产收益率"] = finance["净利润"] / finance["总资产"]
finance["是否盈利"] = finance["净利润"] > 0

finance[["证券代码", "证券简称", "年份", "营业收入_亿元", "净利率", "资产收益率", "是否盈利"]].head()

,证券代码,证券简称,年份,营业收入_亿元,净利率,资产收益率,是否盈利
0,000001,平安银行,2018,1062.120000,0.233665,0.007260,True
1,000001,平安银行,2019,1268.140000,0.222333,0.007158,True
2,000001,平安银行,2020,1432.420000,0.201952,0.006474,True
3,000002,万科A,2018,2976.793311,0.165521,0.032234,True
4,000002,万科A,2019,3678.938775,0.149857,0.031869,True


现在可以得到一个阶段性结果：2020 年盈利且营业收入不缺失的公司，按营业收入排序。

In [6]:
finance_2020_rank = (
    finance
    .loc[
        (finance["年份"] == 2020)
        & finance["是否盈利"]
        & finance["营业收入"].notna(),
        ["证券代码", "证券简称", "营业收入_亿元", "净利率", "资产收益率", "资产负债率"],
    ]
    .sort_values("营业收入_亿元", ascending=False)
)

finance_2020_rank.head(10)

,证券代码,证券简称,营业收入_亿元,净利率,资产收益率,资产负债率
53,000333,美的集团,2857.097290,0.096274,0.076326,0.6553
56,000338,潍柴动力,1974.910929,0.057090,0.041643,0.7029
81,600000,浦发银行,1746.870000,0.337707,0.007420,0.9188
87,600016,民生银行,1667.770000,0.210473,0.005050,0.9221
2,000001,平安银行,1432.420000,0.201952,0.006474,0.9185
84,600015,华夏银行,927.170000,0.232622,0.006344,0.9169
20,000016,深康佳A,503.518366,0.010726,0.010828,0.7851
69,000550,江铃汽车,330.957337,0.016640,0.019539,0.6102
66,000538,云南白药,327.427668,0.168313,0.099802,0.3056
41,000049,德赛电池,193.978245,0.038163,0.081895,0.6878


阶段 1 小结：我们只用一张财务表，已经用到了读取、查看、复制、去重、缺失值检查、类型转换、日期转换、列运算、条件筛选、排序。

## 阶段 2：加入公司信息，让排名结果有行业和地区背景

现在的问题是：财务表只能告诉我们哪家公司收入高，但不能告诉我们它们来自哪些行业和地区。第二阶段读取公司信息表，并把它接到财务结果上。

In [7]:
company_raw = pd.read_excel(
    "data/company_info_example.xlsx",
    converters={"证券代码": read_code},
)

company_raw.head()

,证券代码,证券简称,公司全称,上市市场,行业代码,行业名称,省份,城市,上市日期,成立日期,上市状态
0,000001,平安银行,平安银行股份有限公司,SZSE,J66,货币金融服务,广东省,深圳市,1991-04-03,1987-12-22,正常上市
1,000002,万科A,万科企业股份有限公司,SZSE,K70,房地产业,广东省,深圳市,1991-01-29,1988-11-01,正常上市
2,000004,国华网安,深圳国华网安科技股份有限公司,SZSE,I65,软件和信息技术服务业,广东省,深圳市,1991-01-14,1986-05-05,正常上市
3,000006,深振业A,深圳市振业(集团)股份有限公司,SZSE,K70,NaN,广东省,深圳市,1992-04-27,1989-04-01,正常上市
4,000007,*ST 全新,深圳市全新好股份有限公司,SZSE,K70,房地产业,广东省,深圳市,1992-04-13,1988-11-21,ST


公司信息表也需要简单清洗：文本去空格、日期转换、填补少量缺失。

In [8]:
company = company_raw.copy()

company["证券简称"] = company["证券简称"].str.strip()
company["上市日期"] = pd.to_datetime(company["上市日期"], errors="coerce")
company["成立日期"] = pd.to_datetime(company["成立日期"], errors="coerce")

company[["行业名称", "城市"]] = company[["行业名称", "城市"]].fillna("未知")

company["行业名称"].value_counts().head(10)

行业名称
货币金融服务              4
软件和信息技术服务业          4
计算机、通信和其他电子设备制造业    4
汽车制造业               4
电气机械及器材制造业          4
医药制造业               4
房地产业                3
批发业                 2
未知                  1
Name: count, dtype: int64

把阶段 1 的公司排名和公司信息表合并。合并后检查行数和缺失，确认没有明显匹配失败。

In [9]:
rank_with_info = finance_2020_rank.merge(
    company[["证券代码", "行业名称", "省份", "城市", "上市日期"]],
    on="证券代码",
    how="left",
)

print("合并前行数：", len(finance_2020_rank))
print("合并后行数：", len(rank_with_info))
print("行业缺失数量：", rank_with_info["行业名称"].isna().sum())

rank_with_info.head(10)

合并前行数： 26
合并后行数： 26
行业缺失数量： 0


,证券代码,证券简称,营业收入_亿元,净利率,资产收益率,资产负债率,行业名称,省份,城市,上市日期
0,000333,美的集团,2857.097290,0.096274,0.076326,0.6553,电气机械及器材制造业,广东省,佛山市,2013-09-18
1,000338,潍柴动力,1974.910929,0.057090,0.041643,0.7029,汽车制造业,山东省,潍坊市,2004-03-11
2,600000,浦发银行,1746.870000,0.337707,0.007420,0.9188,货币金融服务,上海市,上海市,1999-11-10
3,600016,民生银行,1667.770000,0.210473,0.005050,0.9221,货币金融服务,北京市,北京市,2000-12-19
4,000001,平安银行,1432.420000,0.201952,0.006474,0.9185,货币金融服务,广东省,深圳市,1991-04-03
5,600015,华夏银行,927.170000,0.232622,0.006344,0.9169,货币金融服务,北京市,北京市,2003-09-12
6,000016,深康佳A,503.518366,0.010726,0.010828,0.7851,计算机、通信和其他电子设备制造业,广东省,深圳市,1992-03-27
7,000550,江铃汽车,330.957337,0.016640,0.019539,0.6102,汽车制造业,江西省,南昌市,1993-12-01
8,000538,云南白药,327.427668,0.168313,0.099802,0.3056,医药制造业,云南省,昆明市,1993-12-15
9,000049,德赛电池,193.978245,0.038163,0.081895,0.6878,电气机械及器材制造业,广东省,深圳市,1995-03-20


现在可以回答更具体的问题：2020 年营业收入最高的公司主要来自哪些行业。

In [10]:
rank_with_info[["证券代码", "证券简称", "行业名称", "省份", "营业收入_亿元", "净利率"]].head(10)

,证券代码,证券简称,行业名称,省份,营业收入_亿元,净利率
0,000333,美的集团,电气机械及器材制造业,广东省,2857.097290,0.096274
1,000338,潍柴动力,汽车制造业,山东省,1974.910929,0.057090
2,600000,浦发银行,货币金融服务,上海市,1746.870000,0.337707
3,600016,民生银行,货币金融服务,北京市,1667.770000,0.210473
4,000001,平安银行,货币金融服务,广东省,1432.420000,0.201952
5,600015,华夏银行,货币金融服务,北京市,927.170000,0.232622
6,000016,深康佳A,计算机、通信和其他电子设备制造业,广东省,503.518366,0.010726
7,000550,江铃汽车,汽车制造业,江西省,330.957337,0.016640
8,000538,云南白药,医药制造业,云南省,327.427668,0.168313
9,000049,德赛电池,电气机械及器材制造业,广东省,193.978245,0.038163


阶段 2 小结：这里带出了第二张表的读取和清洗、`merge`、合并结果检查，以及带背景变量的筛选和展示。

## 阶段 3：从公司名单上升到行业比较

第三阶段不再只看单家公司，而是按行业汇总。目标是得到一张行业层面的分析表。

In [11]:
analysis_df = finance.merge(
    company[["证券代码", "行业名称", "省份"]],
    on="证券代码",
    how="left",
)

analysis_df.head()

,证券代码,证券简称,统计截止日期,年份,总资产,总负债,净资产,营业收入,净利润,资产负债率,总资产_亿元,营业收入_亿元,净利率,资产收益率,是否盈利,行业名称,省份
0,000001,平安银行,2018-12-31,2018,3.418592e+12,3.178550e+12,2.400420e+11,1.062120e+11,2.481800e+10,0.9298,34185.920000,1062.120000,0.233665,0.007260,True,货币金融服务,广东省
1,000001,平安银行,2019-12-31,2019,3.939070e+12,3.626087e+12,3.129830e+11,1.268140e+11,2.819500e+10,0.9205,39390.700000,1268.140000,0.222333,0.007158,True,货币金融服务,广东省
2,000001,平安银行,2020-12-31,2020,4.468514e+12,4.104383e+12,3.641310e+11,1.432420e+11,2.892800e+10,0.9185,44685.140000,1432.420000,0.201952,0.006474,True,货币金融服务,广东省
3,000002,万科A,2018-12-31,2018,1.528579e+12,1.292959e+12,2.356207e+11,2.976793e+11,4.927229e+10,0.8459,15285.793565,2976.793311,0.165521,0.032234,True,房地产业,广东省
4,000002,万科A,2019-12-31,2019,1.729929e+12,1.459350e+12,2.705791e+11,3.678939e+11,5.513161e+10,0.8436,17299.294504,3678.938775,0.149857,0.031869,True,房地产业,广东省


先做 2020 年行业汇总。

In [12]:
industry_2020 = (
    analysis_df[analysis_df["年份"] == 2020]
    .groupby("行业名称")
    .agg(
        公司数=("证券代码", "nunique"),
        平均营业收入_亿元=("营业收入_亿元", "mean"),
        平均净利率=("净利率", "mean"),
        平均资产负债率=("资产负债率", "mean"),
        盈利公司数=("是否盈利", "sum"),
    )
    .sort_values("平均营业收入_亿元", ascending=False)
)

industry_2020

,公司数,平均营业收入_亿元,平均净利率,平均资产负债率,盈利公司数
行业名称,,,,,
货币金融服务,4,1443.557500,0.245688,0.919075,4
电气机械及器材制造业,3,1032.766606,0.048928,0.670800,3
汽车制造业,4,631.454842,0.049060,0.542475,4
计算机、通信和其他电子设备制造业,4,140.555926,0.019264,0.448820,4
医药制造业,4,124.984490,0.103415,0.338025,4
批发业,2,61.544734,0.088032,0.266750,2
软件和信息技术服务业,4,52.403242,0.022898,0.464825,3
未知,1,29.347333,0.307657,0.492200,1
房地产业,3,20.747605,0.178185,0.772333,2


再做一个稍复杂的目标：找出每个行业 2020 年营业收入最高的公司。这里用分组循环展示思路。

In [13]:
top_companies = []

for industry, group in analysis_df[analysis_df["年份"] == 2020].groupby("行业名称"):
    top = group.sort_values("营业收入_亿元", ascending=False).head(1)
    top_companies.append(top)

industry_top_company = pd.concat(top_companies)[
    ["行业名称", "证券代码", "证券简称", "营业收入_亿元", "净利率"]
].sort_values("营业收入_亿元", ascending=False)

industry_top_company

,行业名称,证券代码,证券简称,营业收入_亿元,净利率
53,电气机械及器材制造业,000333,美的集团,2857.097290,0.096274
56,汽车制造业,000338,潍柴动力,1974.910929,0.057090
81,货币金融服务,600000,浦发银行,1746.870000,0.337707
20,计算机、通信和其他电子设备制造业,000016,深康佳A,503.518366,0.010726
66,医药制造业,000538,云南白药,327.427668,0.168313
23,批发业,000019,深粮控股,118.845275,0.033975
72,软件和信息技术服务业,000555,神州信息,106.859768,0.043644
17,房地产业,000011,深物业A,41.043746,0.178185
11,未知,000006,深振业A,29.347333,0.307657


也可以重建一张公司层面的摘要表：每家公司一行，比较 2018 到 2020 年的营业收入变化。

In [14]:
revenue_wide = analysis_df.pivot_table(
    index=["证券代码", "证券简称", "行业名称"],
    columns="年份",
    values="营业收入_亿元",
)

revenue_wide["收入增长率_2018_2020"] = (revenue_wide[2020] / revenue_wide[2018]) - 1

company_summary = (
    revenue_wide
    .reset_index()
    .sort_values("收入增长率_2018_2020", ascending=False)
)

company_summary.head(10)

年份,证券代码,证券简称,行业名称,2018,2019,2020,收入增长率_2018_2020
12,000045,深纺织A,计算机、通信和其他电子设备制造业,12.723568,NaN,21.335748,0.676868
5,000011,深物业A,房地产业,27.872406,39.616699,41.043746,0.472558
11,000030,富奥股份,汽车制造业,78.525364,100.638080,111.134303,0.415266
0,000001,平安银行,货币金融服务,1062.120000,1268.140000,1432.420000,0.348642
28,600015,华夏银行,货币金融服务,694.030000,827.690000,927.170000,0.335922
29,600016,民生银行,货币金融服务,1288.930000,1549.820000,1667.770000,0.293918
18,000338,潍柴动力,汽车制造业,1592.558323,1743.608925,1974.910929,0.240087
22,000538,云南白药,医药制造业,267.082135,296.646739,327.427668,0.225944
21,000513,丽珠集团,医药制造业,88.606557,93.846958,105.204098,0.187317
24,000555,神州信息,软件和信息技术服务业,90.773449,101.460082,106.859768,0.177214


保存最终结果。

In [15]:
industry_2020.to_excel("data/industry_2020_summary.xlsx")
company_summary.to_excel("data/company_summary.xlsx", index=False)

print("已保存行业汇总表和公司摘要表")

已保存行业汇总表和公司摘要表


阶段 3 小结：这里带出了 `groupby().agg()`、排序、分组循环、`concat`、`pivot_table`，以及按分析目标重建数据表。